# Mars lava-tube / skylight candidate detection (JP2)

This notebook scans a **single JP2** image and branches automatically between **IMAGERY** and **DEM/DTM** products.
It generates extensive debug outputs, candidate CSV/GeoJSON, and a mask raster in Drive.

## 1) Mount Google Drive + paths

In [ ]:
from google.colab import drive
import os
import time
from pathlib import Path

# Mount Drive
if not os.path.exists('/content/drive'):
    os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive')

# Fixed project paths
proj = "/content/drive/MyDrive/MARS_LAVATUBES"
hirise_dir = f"{proj}/HIRISE_DTMS"

# Run naming
run_name = time.strftime("%Y%m%d_%H%M%S")
out_dir = f"{proj}/outputs/jp2_scans/{run_name}"
Path(out_dir).mkdir(parents=True, exist_ok=True)

# Default input path (user editable)
input_jp2_path = f"{hirise_dir}/ESP_037232_1770_MIRB.JP2"

print("Project:", proj)
print("HiRISE dir:", hirise_dir)
print("Output dir:", out_dir)
print("Input JP2:", input_jp2_path)

# Validate input
input_path = Path(input_jp2_path)
if not input_path.exists():
    raise FileNotFoundError(
        f"Input JP2 not found: {input_jp2_path}\n"
        "Please update input_jp2_path to a valid JP2 file in Drive."
    )

print("Input size (MB):", round(input_path.stat().st_size / (1024 * 1024), 2))


## 2) Parameter block (edit here)

In [ ]:
# ==========================
# Configuration Parameters
# ==========================

# Input / output
input_jp2_path = input_jp2_path  # editable path
run_name = run_name              # set above
out_dir = out_dir                # set above
band_index = 1                   # choose which band to read (1-based)

# Presets: "NONE" | "HIRISE_IMAGERY" | "CTX_IMAGERY" | "DEM_DTM"
preset = "NONE"

# Preprocessing
seed = 42
blur_sigma = 0.8           # set to 0 to disable
redownsample_if_huge = True
max_pixels = 25_000_000     # auto-downsample if image too large

downsample_factor = 1       # manual downsample factor (>=1)

# Branch override: "AUTO" | "IMAGERY" | "DEM"
branch_override = "AUTO"

# Optional debug: run both branches for troubleshooting
# (saves extra overlays + CSVs)
debug_run_both_branches = False

# Imagery thresholds (dark-pit detection)
min_area = 6
max_area = 8000
min_contrast = 0.03
min_circularity = 0.10
max_eccentricity = 0.99
log_sigma_min = 1.0
log_sigma_max = 12.0
num_sigma = 12
ring_inner = 1.2
ring_outer = 3.0
use_blackhat = True
blackhat_size = 15

# Pixel-scale-aware LoG defaults (AUTO scaling)
expected_diameter_m_min = 20
expected_diameter_m_max = 150
log_sigma_min_user = None  # set to a float to override AUTO sigma
log_sigma_max_user = None

# Detection performance controls
max_blobs = 20000
max_blobs_debug = 5000
max_filter_seconds = 600

# Detection downsample (separate from export)
detect_downsample_factor = 4
export_fullres_mask = False

# Cushing (2012) gates + ranking
shadow_window = 512
shadow_margin = 5.0
trench_method = "FRANGI"  # "FRANGI" or "GABOR"
trench_threshold = 0.2
min_trench_length = 200
trench_distance_px = 12
trench_buffer_px = 40
neighbor_radius_px = 25
neighbor_w = 0.35
incidence_deg = None  # set manually if metadata missing
incidence_deg_fallback = 60.0
use_depth_gate = False
shadow_dark_pctl = 2
shadow_candidate_radius_scale = 1.2
shadow_dark_run_threshold = 0.15
depth_min_m = 10.0
final_selection_mode = "RANK"  # "RANK" or "FILTER"
top_n = 10
nms_radius_factor = 1.5

# Scoring weights
w_dark = 1.0
w_contrast = 1.0
w_edge = 0.5
w_tex = 0.5
w_dist = 0.02
w_nei = 0.35

# DEM thresholds (terrain geometry)
neighborhood_size = 21
min_depth = 0.8
slope_max = 35.0
curvature_threshold = -0.05
curvature_threshold_mode = "PERCENTILE"  # "ABS" or "PERCENTILE"

print("Parameters loaded.")


In [ ]:
# ==========================
# Preset overrides (optional)
# ==========================

if preset == "HIRISE_IMAGERY":
    log_sigma_min = 2
    log_sigma_max = 40
    num_sigma = 12
    min_area = 50
    max_area = 300000
    min_contrast = 0.03
    min_circularity = 0.15
    max_eccentricity = 0.98
    ring_inner = 1.2
    ring_outer = 3.0
    use_blackhat = True
    blackhat_size = 35
elif preset == "CTX_IMAGERY":
    log_sigma_min = 1
    log_sigma_max = 12
    num_sigma = 10
    min_area = 8
    max_area = 15000
    min_contrast = 0.03
    min_circularity = 0.15
    max_eccentricity = 0.98
    ring_inner = 1.2
    ring_outer = 3.0
    use_blackhat = True
    blackhat_size = 15
elif preset == "DEM_DTM":
    curvature_threshold_mode = "PERCENTILE"

if preset != "NONE":
    print(f"Preset applied: {preset}")
    print("Imagery params: log_sigma_min={}, log_sigma_max={}, num_sigma={}, min_area={}, max_area={}, min_contrast={}, min_circularity={}, max_eccentricity={}, ring_inner={}, ring_outer={}, use_blackhat={}, blackhat_size={}"
          .format(log_sigma_min, log_sigma_max, num_sigma, min_area, max_area, min_contrast, min_circularity, max_eccentricity, ring_inner, ring_outer, use_blackhat, blackhat_size))
    print("DEM params: neighborhood_size={}, min_depth={}, slope_max={}, curvature_threshold={}, curvature_threshold_mode={}"
          .format(neighborhood_size, min_depth, slope_max, curvature_threshold, curvature_threshold_mode))


## 3) Imports + dependency helpers

In [ ]:
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import time


def _run(cmd):
    print("\n$", cmd)
    return subprocess.check_call(cmd, shell=True)


def ensure_base_deps():
    try:
        import rasterio  # noqa: F401
        return
    except Exception:
        print("rasterio missing; installing base deps...")
        _run("pip -q install rasterio opencv-python scikit-image scipy matplotlib glymur pillow")


def install_gdal_and_retry():
    print("JP2 driver missing or rasterio failed. Installing GDAL + deps...")
    _run("apt-get -qq update")
    _run("apt-get -qq install -y gdal-bin libgdal-dev")
    _run("pip -q install rasterio opencv-python scikit-image scipy matplotlib glymur pillow")


ensure_base_deps()


## 4) Robust JP2 reading with fallbacks

In [ ]:
import numpy as np


def read_jp2(path, band_index=1, downsample_factor=1):
    # Returns: data (2D float32), transform, crs, tags, nodata, scale_factor
    try:
        import rasterio
        from rasterio.enums import Resampling
    except Exception as exc:
        raise RuntimeError("rasterio import failed even after install") from exc

    try:
        with rasterio.open(path) as src:
            tags = src.tags()
            nodata = src.nodata
            transform = src.transform
            crs = src.crs
            count = src.count
            if band_index < 1 or band_index > count:
                print(f"Band index {band_index} out of range (1..{count}). Using band 1.")
                band_index = 1

            scale_factor = downsample_factor
            if downsample_factor > 1:
                out_shape = (
                    int(src.height / downsample_factor),
                    int(src.width / downsample_factor),
                )
                data = src.read(
                    band_index,
                    out_shape=out_shape,
                    resampling=Resampling.bilinear,
                )
                transform = src.transform * src.transform.scale(
                    (src.width / out_shape[1]),
                    (src.height / out_shape[0])
                )
            else:
                data = src.read(band_index)

        return data.astype(np.float32), transform, crs, tags, nodata, scale_factor

    except Exception as exc:
        print("Rasterio failed:", exc)
        install_gdal_and_retry()
        try:
            with rasterio.open(path) as src:
                tags = src.tags()
                nodata = src.nodata
                transform = src.transform
                crs = src.crs
                count = src.count
                if band_index < 1 or band_index > count:
                    band_index = 1
                if downsample_factor > 1:
                    out_shape = (
                        int(src.height / downsample_factor),
                        int(src.width / downsample_factor),
                    )
                    data = src.read(
                        band_index,
                        out_shape=out_shape,
                        resampling=rasterio.enums.Resampling.bilinear,
                    )
                    transform = src.transform * src.transform.scale(
                        (src.width / out_shape[1]),
                        (src.height / out_shape[0])
                    )
                else:
                    data = src.read(band_index)
            return data.astype(np.float32), transform, crs, tags, nodata, downsample_factor
        except Exception as exc2:
            print("Rasterio retry failed:", exc2)
            print("Trying glymur fallback...")
            import glymur
            jp2 = glymur.Jp2k(path)
            data = jp2[:]
            if data.ndim == 3:
                data = data[:, :, band_index - 1]
            return data.astype(np.float32), None, None, {}, None, downsample_factor


# Auto-downsample if huge
input_path = Path(input_jp2_path)
forced_downsample = downsample_factor
if redownsample_if_huge:
    try:
        import rasterio
        with rasterio.open(input_path) as src:
            total_pixels = src.width * src.height
        if total_pixels > max_pixels and downsample_factor == 1:
            forced_downsample = int(np.ceil(np.sqrt(total_pixels / max_pixels)))
            print(f"Auto downsample factor set to {forced_downsample} for large raster.")
    except Exception:
        pass

# Read data
    t_read_start = time.time()
raw, transform, crs, tags, nodata, scale_factor = read_jp2(
    str(input_path), band_index=band_index, downsample_factor=forced_downsample
)

print("Loaded JP2:", raw.shape, "dtype:", raw.dtype)
print("CRS:", crs)
print("Transform:", transform)
print("Tags:", tags)
print("Band index:", band_index)
print("Scale factor:", scale_factor)
print("Read time (s):", round(time.time() - t_read_start, 2))


## 5) Preprocessing utilities

In [ ]:
from scipy import ndimage
import numpy as np

np.random.seed(seed)


def handle_nodata(data, nodata):
    data = data.astype(np.float32)
    if nodata is not None:
        data = np.where(data == nodata, np.nan, data)
    data = np.where(np.isfinite(data), data, np.nan)
    return data


def normalize_percentile(data, p2=2, p98=98):
    valid = data[np.isfinite(data)]
    if valid.size == 0:
        return data
    lo, hi = np.percentile(valid, (p2, p98))
    if hi - lo < 1e-6:
        return np.clip(data, lo, hi)
    out = (data - lo) / (hi - lo)
    return np.clip(out, 0, 1)


def apply_blur(data, sigma):
    if sigma and sigma > 0:
        return ndimage.gaussian_filter(data, sigma=sigma)
    return data


def quicklook_image(data, is_dem=False):
    if is_dem:
        return normalize_percentile(data)
    return normalize_percentile(data)

## 6) Auto-detect product type

In [ ]:
import numpy as np


def detect_product_type(data, tags, transform, override="AUTO"):
    evidence = []

    if override in ["IMAGERY", "DEM"]:
        return override, [f"Manual override: {override}"]

    tag_text = " ".join([f"{k}:{v}" for k, v in tags.items()]).lower()
    if any(k in tag_text for k in ["dtm", "dem", "elevation", "height"]):
        evidence.append("Tags indicate DEM/DTM")
        return "DEM", evidence

    valid = data[np.isfinite(data)]
    if valid.size == 0:
        evidence.append("No valid data; defaulting to IMAGERY")
        return "IMAGERY", evidence

    vmin, vmax = float(np.nanmin(valid)), float(np.nanmax(valid))
    vstd = float(np.nanstd(valid))
    p2, p98 = np.nanpercentile(valid, [2, 98])
    byte_scaled = (vmin >= 0.0) and (vmax <= 255.0) and (p98 <= 255.0)

    evidence.append(
        f"min={vmin:.2f} max={vmax:.2f} std={vstd:.2f} p2={p2:.2f} p98={p98:.2f} byte_scaled={byte_scaled}"
    )

    if byte_scaled:
        evidence.append("Byte-scaled range detected; classifying as IMAGERY")
        return "IMAGERY", evidence

    # Gradient stats on a manageable 2D sample
    h, w = data.shape
    step = max(1, int(max(h, w) / 512))
    sample = data[::step, ::step]
    gy, gx = np.gradient(sample.astype(np.float32))
    grad_mag = np.sqrt(gx ** 2 + gy ** 2)
    grad_med = float(np.nanmedian(grad_mag))
    grad_p95 = float(np.nanpercentile(grad_mag, 95))
    evidence.append(f"grad_step={step} grad_median={grad_med:.4f} grad_p95={grad_p95:.4f}")

    range_val = vmax - vmin
    if range_val > 100 and vstd > 5:
        evidence.append("Range/std suggest DEM/DTM")
        return "DEM", evidence

    evidence.append("Defaulting to IMAGERY (uncertain)")
    return "IMAGERY", evidence


product_type, evidence = detect_product_type(raw, tags, transform, override=branch_override)

if branch_override == "AUTO" and product_type == "DEM" and np.nanmax(raw) <= 255:
    print("Warning: AUTO selected DEM but max<=255. Switching to IMAGERY. Set branch_override to DEM to force.")
    product_type = "IMAGERY"

print("Detected product type:", product_type)
print("Evidence:")
for e in evidence:
    print(" -", e)


## 7) Preprocess input

In [ ]:
start_time = time.time()

raw = handle_nodata(raw, nodata)

t_pre_start = time.time()
if product_type == "DEM":
    dem_data = raw.copy()
    dem_vis = normalize_percentile(dem_data)
else:
    img_data = raw.copy()
    img_vis = normalize_percentile(img_data)

if product_type == "DEM":
    dem_data = apply_blur(dem_data, blur_sigma)
else:
    img_data = apply_blur(img_data, blur_sigma)

px_m = abs(transform.a) if transform is not None else None
print("Pixel size (m/px):", px_m)
print("Preprocess time (s):", round(time.time() - t_pre_start, 2))

plt.figure(figsize=(6, 6))
plt.imshow(dem_vis if product_type == "DEM" else img_vis, cmap='gray')
plt.title("Input quicklook (normalized)")
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{out_dir}/quicklook.png", dpi=200)
plt.close()

plt.figure(figsize=(6, 4))
vals = raw[np.isfinite(raw)].ravel()
plt.hist(vals, bins=200, color='steelblue', alpha=0.8)
plt.title("Value histogram")
plt.xlabel("Value")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"{out_dir}/histogram.png", dpi=200)
plt.close()

print("Quicklook + histogram saved.")


## 8) DEM/DTM branch: terrain-geometry detection

In [ ]:
from scipy import ndimage
from skimage import measure, morphology


def dem_pipeline(dem, transform):
    print("Running DEM pipeline...")
    print(
        f"Params: neighborhood_size={neighborhood_size}, min_depth={min_depth}, "
        f"slope_max={slope_max}, curvature_threshold={curvature_threshold}, "
        f"curvature_threshold_mode={curvature_threshold_mode}, min_area={min_area}"
    )

    if transform is not None:
        pixel_x = abs(transform.a)
        pixel_y = abs(transform.e)
    else:
        pixel_x = pixel_y = 1.0

    gy, gx = np.gradient(dem, pixel_y, pixel_x)
    slope = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))

    curvature = ndimage.laplace(dem)
    curv_p1, curv_p5, curv_p10 = np.nanpercentile(curvature, [1, 5, 10])

    if curvature_threshold_mode == "PERCENTILE":
        curvature_thresh = curv_p5
        print(f"Curvature threshold (PERCENTILE 5%): {curvature_thresh:.6f}")
    else:
        curvature_thresh = curvature_threshold
        print(f"Curvature threshold (ABS): {curvature_thresh:.6f}")

    size = neighborhood_size
    local_mean = ndimage.uniform_filter(dem, size=size, mode='nearest')
    depth = local_mean - dem

    local_min = (dem == ndimage.minimum_filter(dem, size=size))
    depth_mask = depth >= min_depth
    slope_mask = slope <= slope_max
    curve_mask = curvature <= curvature_thresh

    pit_mask = local_min & depth_mask & slope_mask & curve_mask

    print("Local minima count:", int(local_min.sum()))
    print("After depth filter:", int((local_min & depth_mask).sum()))
    print("After slope filter:", int((local_min & depth_mask & slope_mask).sum()))
    print("After curvature filter:", int(pit_mask.sum()))

    labels = measure.label(pit_mask)
    props = measure.regionprops(labels)

    candidates = []
    for p in props:
        if p.area < min_area:
            continue
        y, x = p.centroid
        score = float(depth[int(y), int(x)])
        candidates.append({
            "id": len(candidates) + 1,
            "x_px": float(x),
            "y_px": float(y),
            "score": float(score),
            "area": float(p.area) * (scale_factor ** 2),
            "depth": float(depth[int(y), int(x)]),
            "slope": float(slope[int(y), int(x)]),
            "curvature": float(curvature[int(y), int(x)])
        })

    plt.figure(figsize=(6, 6))
    plt.imshow(slope, cmap='inferno')
    plt.title('Slope')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/slope.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(curvature, cmap='coolwarm')
    plt.title('Curvature')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/curvature.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(depth, cmap='viridis')
    plt.title('Depth')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/depth.png", dpi=200)
    plt.close()

    return pit_mask, candidates, slope, curvature, depth


## 9) IMAGERY branch: photometric pit/skylight detection

In [ ]:
from skimage import feature, morphology, measure, filters
from skimage.morphology import skeletonize
from scipy import ndimage


def compute_contrast(image, y, x, radius, ring_inner, ring_outer):
    yy, xx = np.ogrid[:image.shape[0], :image.shape[1]]
    dist = np.sqrt((yy - y) ** 2 + (xx - x) ** 2)
    inner = dist <= radius
    ring = (dist >= radius * ring_inner) & (dist <= radius * ring_outer)
    if np.any(inner) and np.any(ring):
        return float(np.nanmean(image[ring]) - np.nanmean(image[inner]))
    return 0.0


def _save_blob_overlay(image, blobs, title, out_path, color='red'):
    plt.figure(figsize=(7, 7))
    plt.imshow(image, cmap='gray')
    for (y, x, sigma) in blobs:
        plt.plot(x, y, '+', color=color, markersize=6)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def _resolve_sigma_range(px_m):
    if log_sigma_min_user is not None and log_sigma_max_user is not None:
        return float(log_sigma_min_user), float(log_sigma_max_user), "USER"

    if px_m is not None:
        diameter_px_min = expected_diameter_m_min / px_m
        diameter_px_max = expected_diameter_m_max / px_m
        sigma_min = max(1.0, (diameter_px_min / 2.0) / 1.5)
        sigma_max = max(sigma_min + 1.0, (diameter_px_max / 2.0) / 1.5)
        sigma_max = min(sigma_max, 80.0)
        print("AUTO sigma scaling: px_m={}, diameter_px_min={:.2f}, diameter_px_max={:.2f}".format(
            px_m, diameter_px_min, diameter_px_max
        ))
        return sigma_min, sigma_max, "AUTO"

    return log_sigma_min, log_sigma_max, "DEFAULT"


def _safe_window_bounds(y, x, half, shape):
    y0 = max(0, y - half)
    y1 = min(shape[0], y + half)
    x0 = max(0, x - half)
    x1 = min(shape[1], x + half)
    return y0, y1, x0, x1


def _estimate_incidence_deg(tags):
    for key in ["INCIDENCE_ANGLE", "INCIDENCE", "INCIDENCE_DEG"]:
        if key in tags:
            try:
                return float(tags[key])
            except Exception:
                continue
    return None


def _detect_trenches(image):
    if trench_method == "FRANGI":
        vessel = filters.frangi(1.0 - image)
    else:
        gabor_real, gabor_imag = filters.gabor(1.0 - image, frequency=0.1)
        vessel = np.hypot(gabor_real, gabor_imag)

    vessel = (vessel - np.nanmin(vessel)) / (np.nanmax(vessel) - np.nanmin(vessel) + 1e-6)
    trench_mask = vessel > trench_threshold
    trench_skeleton = skeletonize(trench_mask)
    return vessel, trench_mask, trench_skeleton


def _dark_run_length(image, center_y, center_x, dark_thresh):
    directions = [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1)
    ]
    max_len = 0
    for dy, dx in directions:
        length = 0
        y, x = center_y, center_x
        while 0 <= y < image.shape[0] and 0 <= x < image.shape[1]:
            if image[y, x] <= dark_thresh:
                length += 1
                y += dy
                x += dx
            else:
                break
        max_len = max(max_len, length)
    return max_len


def _apply_nms(candidates_list, radius_factor):
    keep = []
    for cand in sorted(candidates_list, key=lambda c: c["score"], reverse=True):
        too_close = False
        for kept in keep:
            dy = cand["y_px_ds"] - kept["y_px_ds"]
            dx = cand["x_px_ds"] - kept["x_px_ds"]
            radius = radius_factor * max(cand.get("radius_ds", 1.0), kept.get("radius_ds", 1.0))
            if np.hypot(dx, dy) <= radius:
                too_close = True
                break
        if not too_close:
            keep.append(cand)
    return keep


def _print_metric_percentiles(name, values):
    if not values:
        print(f"{name} percentiles: n/a")
        return
    arr = np.array(values, dtype=float)
    if np.all(np.isnan(arr)):
        print(f"{name} percentiles: n/a")
        return
    p10, p50, p90 = np.nanpercentile(arr, [10, 50, 90])
    print(f"{name} percentiles: p10={p10:.4f}, p50={p50:.4f}, p90={p90:.4f}")


def imagery_pipeline(image):
    print("Running IMAGERY pipeline...")
    contrast_time = 0.0
    shape_time = 0.0
    print(
        "Params: min_area={}, max_area={}, min_contrast={}, min_circularity={}, "
        "max_eccentricity={}, ring_inner={}, ring_outer={}, use_blackhat={}, blackhat_size={}, scale_factor={}, detect_downsample_factor={}, max_blobs={}, max_filter_seconds={}".format(
            min_area, max_area, min_contrast, min_circularity, max_eccentricity,
            ring_inner, ring_outer, use_blackhat, blackhat_size, scale_factor, detect_downsample_factor, max_blobs, max_filter_seconds
        )
    )

    t_downsample = time.time()
    img = normalize_percentile(image)

    if use_blackhat:
        selem = morphology.disk(blackhat_size)
        bh = morphology.black_tophat(img, selem)
        img_proc = normalize_percentile(bh)
    else:
        bh = None
        img_proc = img

    detect_img = img_proc[::detect_downsample_factor, ::detect_downsample_factor]
    print("Detection downsampled shape:", detect_img.shape)
    print("Downsample time (s):", round(time.time() - t_downsample, 2))

    sigma_min, sigma_max, sigma_source = _resolve_sigma_range(px_m)
    print("LoG sigma range ({}): min={:.2f}, max={:.2f}".format(sigma_source, sigma_min, sigma_max))

    t_log = time.time()
    inv = 1.0 - detect_img
    blobs = feature.blob_log(
        inv,
        min_sigma=sigma_min,
        max_sigma=sigma_max,
        num_sigma=num_sigma,
        threshold=0.03
    )
    print("Blobs detected (LoG):", len(blobs))
    print("LoG detect time (s):", round(time.time() - t_log, 2))

    _save_blob_overlay(detect_img, blobs, "Blobs (raw LoG)", f"{out_dir}/blobs_overlay_raw.png")

    t_response = time.time()
    sigmas = np.linspace(sigma_min, sigma_max, num_sigma)
    response = np.zeros_like(detect_img, dtype=np.float32)
    for s in sigmas:
        resp = -ndimage.gaussian_laplace(detect_img, s)
        response = np.maximum(response, resp)

    # Contrast map (local difference)
    small = ndimage.uniform_filter(detect_img, size=max(3, int(ring_inner * 4)))
    large = ndimage.uniform_filter(detect_img, size=max(5, int(ring_outer * 8)))
    contrast_map = large - small
    print("Response/contrast map time (s):", round(time.time() - t_response, 2))

    # Top-K cap on blobs
    t_topk = time.time()
    if blobs.size > 0:
        ys = np.clip(np.round(blobs[:, 0]).astype(int), 0, response.shape[0] - 1)
        xs = np.clip(np.round(blobs[:, 1]).astype(int), 0, response.shape[1] - 1)
        scores = response[ys, xs]
        blob_scores = np.stack([blobs[:, 0], blobs[:, 1], blobs[:, 2], scores], axis=1)
    else:
        blob_scores = np.zeros((0, 4), dtype=float)

    cap = max_blobs_debug if debug_run_both_branches else max_blobs
    if blob_scores.shape[0] > cap:
        order = np.argsort(blob_scores[:, 3])[::-1][:cap]
        blob_scores = blob_scores[order]
        print(f"Top-K applied: kept {cap} blobs (from {len(blobs)})")
    else:
        print("Top-K not needed.")

    blobs = blob_scores[:, :3]
    print("Top-K filter time (s):", round(time.time() - t_topk, 2))

    candidates = []
    mask = np.zeros_like(detect_img, dtype=bool)

    area_pass = 0
    contrast_pass = 0
    shape_pass = 0

    blobs_area_pass = []
    blobs_contrast_pass = []
    blobs_shape_pass = []
    contrast_values = []

    t_filter = time.time()
    for i, (y, x, sigma) in enumerate(blobs, start=1):
        if (i % 2000) == 0:
            elapsed = time.time() - t_filter
            rate = i / elapsed if elapsed > 0 else 0
            remaining = (len(blobs) - i) / rate if rate > 0 else float('inf')
            print(f"Processed {i}/{len(blobs)} blobs | elapsed={elapsed:.1f}s | est remaining={remaining:.1f}s")

        if (time.time() - t_filter) > max_filter_seconds:
            print("Stopped early due to time budget")
            break

        radius = sigma * np.sqrt(2)
        y = int(round(y))
        x = int(round(x))

        if y <= 0 or x <= 0 or y >= detect_img.shape[0] - 1 or x >= detect_img.shape[1] - 1:
            continue

        area_ds = np.pi * (radius ** 2)
        if area_ds < min_area or area_ds > max_area:
            continue
        area_pass += 1
        blobs_area_pass.append((y, x, sigma))

        t_contrast = time.time()
        contrast = compute_contrast(detect_img, y, x, radius, ring_inner, ring_outer)
        contrast_time += time.time() - t_contrast
        contrast_values.append(contrast)
        if contrast < min_contrast:
            continue
        contrast_pass += 1
        blobs_contrast_pass.append((y, x, sigma))

        t_shape = time.time()
        rr, cc = np.ogrid[:detect_img.shape[0], :detect_img.shape[1]]
        circle = (rr - y) ** 2 + (cc - x) ** 2 <= radius ** 2
        label = measure.label(circle)
        props = measure.regionprops(label)
        shape_time += time.time() - t_shape
        if not props:
            continue
        p = props[0]

        circularity = 4 * np.pi * p.area / (p.perimeter ** 2 + 1e-6)
        eccentricity = p.eccentricity

        if circularity < min_circularity or eccentricity > max_eccentricity:
            continue
        shape_pass += 1
        blobs_shape_pass.append((y, x, sigma))

        score = float(contrast * circularity * np.sqrt(area_ds))
        area_report = float(area_ds) * (scale_factor ** 2) * (detect_downsample_factor ** 2)

        x_data = float(x) * detect_downsample_factor
        y_data = float(y) * detect_downsample_factor
        x_full = x_data * scale_factor
        y_full = y_data * scale_factor

        mask |= circle
        candidates.append({
            "id": len(candidates) + 1,
            "x_px": x_data,
            "y_px": y_data,
            "x_px_ds": float(x),
            "y_px_ds": float(y),
            "x_px_full": x_full,
            "y_px_full": y_full,
            "score": float(score),
            "area": area_report,
            "circularity": float(circularity),
            "eccentricity": float(eccentricity),
            "contrast": float(contrast),
            "radius_ds": float(radius)
        })

    print("After area filter:", area_pass)
    print("After contrast filter:", contrast_pass)
    print("After shape filter:", shape_pass)
    print("Filtering time (s):", round(time.time() - t_filter, 2))
    print("Contrast filter time (s):", round(contrast_time, 2))
    print("Shape filter time (s):", round(shape_time, 2))

    _save_blob_overlay(detect_img, blobs_area_pass, "Blobs (area pass)", f"{out_dir}/blobs_overlay_area_pass.png", color='orange')
    _save_blob_overlay(detect_img, blobs_contrast_pass, "Blobs (contrast pass)", f"{out_dir}/blobs_overlay_contrast_pass.png", color='yellow')
    _save_blob_overlay(detect_img, blobs_shape_pass, "Blobs (shape pass)", f"{out_dir}/blobs_overlay_shape_pass.png", color='lime')

    if contrast_values:
        plt.figure(figsize=(6, 4))
        plt.hist(contrast_values, bins=50, color='slateblue', alpha=0.8)
        p10, p50, p90 = np.percentile(contrast_values, [10, 50, 90])
        plt.title(f"Contrast values (p10={p10:.3f}, p50={p50:.3f}, p90={p90:.3f})")
        plt.xlabel("Contrast")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(f"{out_dir}/contrast_histogram.png", dpi=200)
        plt.close()
    else:
        print("No contrast values to plot.")

    plt.figure(figsize=(6, 6))
    plt.imshow(response, cmap='viridis')
    plt.title("LoG response")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/log_response.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(contrast_map, cmap='magma')
    plt.title("Contrast map")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/contrast_map.png", dpi=200)
    plt.close()

    if bh is not None:
        plt.figure(figsize=(6, 6))
        plt.imshow(bh, cmap='gray')
        plt.title("Black-hat output")
        plt.colorbar()
        plt.tight_layout()
        plt.savefig(f"{out_dir}/blackhat.png", dpi=200)
        plt.close()

    # Cushing (2012) gates
    t_gates = time.time()
    incidence_meta = _estimate_incidence_deg(tags)
    incidence_use = incidence_deg if incidence_deg is not None else incidence_meta
    if use_depth_gate and incidence_use is None:
        incidence_use = incidence_deg_fallback
        print(f"Warning: incidence angle missing; using fallback {incidence_use} deg for depth proxy.")
    elif incidence_use is None:
        print("Warning: incidence angle missing; depth proxy disabled.")

    vessel, trench_mask, trench_skeleton = _detect_trenches(detect_img)
    trench_buffer = morphology.binary_dilation(trench_skeleton, morphology.disk(trench_buffer_px))

    plt.figure(figsize=(7, 7))
    plt.imshow(detect_img, cmap='gray')
    plt.imshow(trench_mask, cmap='Blues', alpha=0.35)
    plt.contour(trench_skeleton, colors='lime', linewidths=0.5)
    plt.contour(trench_buffer, colors='yellow', linewidths=0.5)
    plt.title("Trench mask + skeleton + buffer")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/trench_overlay.png", dpi=200)
    plt.close()

    trench_dist = ndimage.distance_transform_edt(~trench_skeleton)

    gx = ndimage.sobel(detect_img, axis=1)
    gy = ndimage.sobel(detect_img, axis=0)

    gated_candidates = []
    px_m_detect = px_m * detect_downsample_factor if px_m is not None else None

    darkness_vals = []
    contrast_vals = []
    edge_vals = []
    texture_vals = []
    depth_vals = []
    neighbor_vals = []

    for cand in candidates:
        cy = int(round(cand["y_px_ds"]))
        cx = int(round(cand["x_px_ds"]))
        radius = max(2, int(round(cand["radius_ds"] * shadow_candidate_radius_scale)))

        if not trench_buffer[cy, cx]:
            continue

        y0, y1, x0, x1 = _safe_window_bounds(cy, cx, shadow_window // 2, detect_img.shape)
        local = detect_img[y0:y1, x0:x1]
        if local.size == 0:
            continue

        p2 = np.nanpercentile(local, shadow_dark_pctl)
        shadow_mask = local <= p2
        if not np.any(shadow_mask):
            continue
        shadow_ref = float(np.nanmedian(local[shadow_mask]))

        rr, cc = np.ogrid[:detect_img.shape[0], :detect_img.shape[1]]
        circle = (rr - cy) ** 2 + (cc - cx) ** 2 <= radius ** 2
        center_vals = detect_img[circle] if np.any(circle) else np.array([detect_img[cy, cx]])
        center_min = float(np.nanmin(center_vals))
        center_mean = float(np.nanmean(center_vals))
        ring_mask = (rr - cy) ** 2 + (cc - cx) ** 2 <= (radius * ring_outer) ** 2
        ring_mask &= (rr - cy) ** 2 + (cc - cx) ** 2 >= (radius * ring_inner) ** 2
        ring_vals = detect_img[ring_mask] if np.any(ring_mask) else np.array([center_mean])
        ring_mean = float(np.nanmean(ring_vals))
        darkness_delta = ring_mean - center_min
        ring_contrast = (ring_mean - center_mean) / (ring_mean + 1e-6)

        edge_strength = 0.0
        if np.any(ring_mask):
            edge_strength = float(np.nanmean(np.hypot(gx[ring_mask], gy[ring_mask])))

        texture_std = float(np.nanstd(center_vals))

        depth_proxy_m = float('nan')
        Ds_m = None
        if use_depth_gate and incidence_use is not None and px_m_detect is not None:
            dark_thresh = shadow_ref + shadow_dark_run_threshold
            dark_run = _dark_run_length(detect_img, cy, cx, dark_thresh)
            Ds_m = dark_run * px_m_detect
            if incidence_use > 0:
                depth_proxy_m = Ds_m / np.tan(np.deg2rad(incidence_use))

        cand.update({
            "shadow_ref": shadow_ref,
            "candidate_min": center_min,
            "darkness_delta": darkness_delta,
            "ring_mean": ring_mean,
            "ring_contrast": ring_contrast,
            "trench_distance_px": float(trench_dist[cy, cx]),
            "Ds_m": Ds_m,
            "incidence_deg": incidence_use,
            "d_min_m": depth_proxy_m,
            "edge_strength": edge_strength,
            "texture_std": texture_std
        })

        if final_selection_mode == "FILTER":
            if center_min >= shadow_ref - shadow_margin:
                continue
            if trench_dist[cy, cx] > trench_distance_px:
                continue
            local_skeleton = trench_skeleton[y0:y1, x0:x1]
            if local_skeleton.sum() < min_trench_length:
                continue
            if use_depth_gate and depth_proxy_m == depth_proxy_m and depth_proxy_m < depth_min_m:
                continue

        darkness_vals.append(darkness_delta)
        contrast_vals.append(ring_contrast)
        edge_vals.append(edge_strength)
        texture_vals.append(texture_std)
        depth_vals.append(depth_proxy_m if depth_proxy_m == depth_proxy_m else float('nan'))

        cand["neighbor_count"] = 0
        gated_candidates.append(cand)

    print("After trench buffer:", len(gated_candidates))

    if len(gated_candidates) == 0 and len(candidates) > 0:
        print("Warning: trench buffer removed all candidates; rerunning without buffer.")
        gated_candidates = []
        darkness_vals = []
        contrast_vals = []
        edge_vals = []
        texture_vals = []
        depth_vals = []
        neighbor_vals = []
        for cand in candidates:
            cy = int(round(cand["y_px_ds"]))
            cx = int(round(cand["x_px_ds"]))
            radius = max(2, int(round(cand["radius_ds"] * shadow_candidate_radius_scale)))

            y0, y1, x0, x1 = _safe_window_bounds(cy, cx, shadow_window // 2, detect_img.shape)
            local = detect_img[y0:y1, x0:x1]
            if local.size == 0:
                continue

            p2 = np.nanpercentile(local, shadow_dark_pctl)
            shadow_mask = local <= p2
            if not np.any(shadow_mask):
                continue
            shadow_ref = float(np.nanmedian(local[shadow_mask]))

            rr, cc = np.ogrid[:detect_img.shape[0], :detect_img.shape[1]]
            circle = (rr - cy) ** 2 + (cc - cx) ** 2 <= radius ** 2
            center_vals = detect_img[circle] if np.any(circle) else np.array([detect_img[cy, cx]])
            center_min = float(np.nanmin(center_vals))
            center_mean = float(np.nanmean(center_vals))
            ring_mask = (rr - cy) ** 2 + (cc - cx) ** 2 <= (radius * ring_outer) ** 2
            ring_mask &= (rr - cy) ** 2 + (cc - cx) ** 2 >= (radius * ring_inner) ** 2
            ring_vals = detect_img[ring_mask] if np.any(ring_mask) else np.array([center_mean])
            ring_mean = float(np.nanmean(ring_vals))
            darkness_delta = ring_mean - center_min
            ring_contrast = (ring_mean - center_mean) / (ring_mean + 1e-6)

            edge_strength = 0.0
            if np.any(ring_mask):
                edge_strength = float(np.nanmean(np.hypot(gx[ring_mask], gy[ring_mask])))

            texture_std = float(np.nanstd(center_vals))

            depth_proxy_m = float('nan')
            Ds_m = None
            if use_depth_gate and incidence_use is not None and px_m_detect is not None:
                dark_thresh = shadow_ref + shadow_dark_run_threshold
                dark_run = _dark_run_length(detect_img, cy, cx, dark_thresh)
                Ds_m = dark_run * px_m_detect
                if incidence_use > 0:
                    depth_proxy_m = Ds_m / np.tan(np.deg2rad(incidence_use))

            cand.update({
                "shadow_ref": shadow_ref,
                "candidate_min": center_min,
                "darkness_delta": darkness_delta,
                "ring_mean": ring_mean,
                "ring_contrast": ring_contrast,
                "trench_distance_px": float(trench_dist[cy, cx]),
                "Ds_m": Ds_m,
                "incidence_deg": incidence_use,
                "d_min_m": depth_proxy_m,
                "edge_strength": edge_strength,
                "texture_std": texture_std
            })

            darkness_vals.append(darkness_delta)
            contrast_vals.append(ring_contrast)
            edge_vals.append(edge_strength)
            texture_vals.append(texture_std)
            depth_vals.append(depth_proxy_m if depth_proxy_m == depth_proxy_m else float('nan'))

            cand["neighbor_count"] = 0
            gated_candidates.append(cand)
    # Neighbor counts for clustering
    coords = np.array([[c["y_px_ds"], c["x_px_ds"]] for c in gated_candidates]) if gated_candidates else np.empty((0, 2))
    for i, cand in enumerate(gated_candidates):
        if coords.size == 0:
            cand["neighbor_count"] = 0
            continue
        dists = np.hypot(coords[:, 0] - cand["y_px_ds"], coords[:, 1] - cand["x_px_ds"])
        cand["neighbor_count"] = int(np.sum(dists <= neighbor_radius_px)) - 1
        neighbor_vals.append(cand["neighbor_count"])

    for cand in gated_candidates:
        depth_bonus = (cand["d_min_m"] / 10.0) if cand["d_min_m"] == cand["d_min_m"] else 0.0
        score = (
            w_dark * cand["darkness_delta"]
            + w_contrast * cand["ring_contrast"]
            + w_edge * cand["edge_strength"]
            - w_tex * cand["texture_std"]
            - w_dist * cand["trench_distance_px"]
            + w_nei * cand["neighbor_count"]
            + depth_bonus
        )
        cand["score"] = float(score)

        print(f"Gate metrics id={cand['id']}: center_min={cand['candidate_min']:.4f}, ring_mean={cand['ring_mean']:.4f}, delta={cand['darkness_delta']:.4f}")

    print("Gate pass count:", len(gated_candidates))
    print("Gate time (s):", round(time.time() - t_gates, 2))

    _print_metric_percentiles("darkness_delta", darkness_vals)
    _print_metric_percentiles("ring_contrast", contrast_vals)
    _print_metric_percentiles("edge_strength", edge_vals)
    _print_metric_percentiles("texture_std", texture_vals)
    _print_metric_percentiles("depth_proxy_m", depth_vals)
    _print_metric_percentiles("neighbor_count", neighbor_vals)

    # Non-maximum suppression + top-N
    pre_nms = len(gated_candidates)
    final_candidates = _apply_nms(gated_candidates, nms_radius_factor)
    post_nms = len(final_candidates)
    if final_selection_mode == "RANK":
        final_candidates = sorted(final_candidates, key=lambda c: c["score"], reverse=True)[:top_n]
    print(f"Candidates before NMS: {pre_nms}, after NMS: {post_nms}, final: {len(final_candidates)}")

    # Overlay final candidates on trench map
    plt.figure(figsize=(7, 7))
    plt.imshow(detect_img, cmap='gray')
    plt.imshow(trench_mask, cmap='Blues', alpha=0.35)
    plt.contour(trench_skeleton, colors='lime', linewidths=0.5)
    plt.contour(trench_buffer, colors='yellow', linewidths=0.5)
    for c in final_candidates:
        plt.plot(c["x_px_ds"], c["y_px_ds"], 'r+', markersize=8)
        plt.text(c["x_px_ds"] + 2, c["y_px_ds"] + 2, str(c["id"]), color='yellow', fontsize=8)
    plt.title("Final candidates with trench context")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/trench_candidates_overlay.png", dpi=200)
    plt.close()

    # Top-N overlay on quicklook
    plt.figure(figsize=(7, 7))
    plt.imshow(detect_img, cmap='gray')
    for c in final_candidates:
        plt.plot(c["x_px_ds"], c["y_px_ds"], 'r+', markersize=8)
        plt.text(c["x_px_ds"] + 2, c["y_px_ds"] + 2, str(c["id"]), color='yellow', fontsize=8)
    plt.title("Top-N candidates (detection grid)")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/candidates_overlay_topN.png", dpi=200)
    plt.close()

    # Write debug counts
    with open(f"{out_dir}/debug_counts.txt", "w") as f:
        f.write(f"Blobs detected: {len(blobs)}\n")
        f.write(f"After area filter: {area_pass}\n")
        f.write(f"After contrast filter: {contrast_pass}\n")
        f.write(f"After shape filter: {shape_pass}\n")
        f.write(f"After trench buffer: {len(gated_candidates)}\n")
        f.write(f"Candidates before NMS: {pre_nms}\n")
        f.write(f"Candidates after NMS: {post_nms}\n")
        f.write(f"Final top_n: {len(final_candidates)}\n")
        f.write(f"detect_downsample_factor: {detect_downsample_factor}\n")
        f.write(f"log_sigma_min: {log_sigma_min}\n")
        f.write(f"log_sigma_max: {log_sigma_max}\n")
        f.write(f"num_sigma: {num_sigma}\n")
        f.write(f"min_area: {min_area}\n")
        f.write(f"max_area: {max_area}\n")
        f.write(f"min_contrast: {min_contrast}\n")
        f.write(f"min_circularity: {min_circularity}\n")
        f.write(f"max_eccentricity: {max_eccentricity}\n")
        f.write(f"ring_inner: {ring_inner}\n")
        f.write(f"ring_outer: {ring_outer}\n")
        f.write(f"trench_buffer_px: {trench_buffer_px}\n")
        f.write(f"neighbor_radius_px: {neighbor_radius_px}\n")
        f.write(f"neighbor_w: {neighbor_w}\n")
        f.write(f"final_selection_mode: {final_selection_mode}\n")
        f.write(f"top_n: {top_n}\n")

    return mask, final_candidates, response


## 10) Run detection branch + overlay

In [ ]:
t_branch = time.time()
if product_type == "DEM":
    pit_mask, candidates, slope, curvature, depth = dem_pipeline(dem_data, transform)
    branch = "DEM"
else:
    pit_mask, candidates, response = imagery_pipeline(img_data)
    branch = "IMAGERY"
print("Branch runtime (s):", round(time.time() - t_branch, 2))

# Optional debug: run both branches regardless of selection
if debug_run_both_branches:
    print("Debug mode: running both IMAGERY and DEM pipelines.")
    dem_mask_dbg, dem_candidates_dbg, slope_dbg, curvature_dbg, depth_dbg = dem_pipeline(dem_data, transform)
    img_mask_dbg, img_candidates_dbg, response_dbg = imagery_pipeline(img_data)

    # Save debug overlays
    plt.figure(figsize=(7, 7))
    plt.imshow(dem_vis, cmap='gray')
    for c in dem_candidates_dbg:
        plt.plot(c["x_px"], c["y_px"], 'r+', markersize=8)
        plt.text(c["x_px"] + 2, c["y_px"] + 2, str(c["id"]), color='yellow', fontsize=8)
    plt.title("Candidates overlay (DEM)")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/candidates_overlay_DEM.png", dpi=200)
    plt.close()

    plt.figure(figsize=(7, 7))
    plt.imshow(img_vis, cmap='gray')
    for c in img_candidates_dbg:
        plt.plot(c["x_px"], c["y_px"], 'r+', markersize=8)
        plt.text(c["x_px"] + 2, c["y_px"] + 2, str(c["id"]), color='yellow', fontsize=8)
    plt.title("Candidates overlay (IMAGERY)")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/candidates_overlay_IMAGERY.png", dpi=200)
    plt.close()

    debug_candidates = {
        "DEM": dem_candidates_dbg,
        "IMAGERY": img_candidates_dbg
    }
else:
    debug_candidates = None

print("Total candidates:", len(candidates))

plt.figure(figsize=(7, 7))
base = dem_vis if product_type == "DEM" else img_vis
plt.imshow(base, cmap='gray')
for c in candidates:
    plt.plot(c["x_px"], c["y_px"], 'r+', markersize=8)
    plt.text(c["x_px"] + 2, c["y_px"] + 2, str(c["id"]), color='yellow', fontsize=8)
plt.title(f"Candidates overlay ({branch})")
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{out_dir}/candidates_overlay.png", dpi=200)
plt.close()


## 11) Export outputs (CSV, GeoJSON, mask raster)

In [ ]:
import csv
import json


t_export = time.time()

def write_candidates_csv(path, candidates_list, branch_name):
    base_fields = [
        "candidate_id", "x_full", "y_full", "x_px_ds", "y_px_ds", "score", "area",
        "circularity", "eccentricity", "contrast", "branch",
        "diameter_m_est", "shadow_ref", "candidate_min", "darkness_delta",
        "ring_mean", "ring_contrast", "trench_distance_px", "neighbor_count",
        "Ds_m", "incidence_deg", "d_min_m", "edge_strength", "texture_std"
    ]
    extra_fields = []
    if branch_name == "DEM":
        extra_fields = ["depth", "slope", "curvature"]

    fields = base_fields + extra_fields

    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for c in candidates_list:
            row = {k: c.get(k, "") for k in fields}
            x_ds = c.get("x_px_ds", c.get("x_px", 0.0))
            y_ds = c.get("y_px_ds", c.get("y_px", 0.0))
            if "x_px_full" in c and "y_px_full" in c:
                x_full = c["x_px_full"]
                y_full = c["y_px_full"]
            else:
                x_full = float(x_ds) * scale_factor
                y_full = float(y_ds) * scale_factor

            row["candidate_id"] = c.get("id", "")
            row["x_full"] = x_full
            row["y_full"] = y_full
            row["x_px_ds"] = x_ds
            row["y_px_ds"] = y_ds
            row["branch"] = branch_name
            row.setdefault("circularity", "")
            row.setdefault("eccentricity", "")
            row.setdefault("contrast", "")

            if px_m is not None:
                diameter_m = c.get("radius_ds", 0.0) * 2.0 * px_m * detect_downsample_factor
            else:
                diameter_m = ""
            row["diameter_m_est"] = diameter_m

            writer.writerow(row)


csv_path = f"{out_dir}/candidates.csv"
write_candidates_csv(csv_path, candidates, branch)
print("Saved:", csv_path)

# Top-N CSV
if branch == "IMAGERY":
    topn_path = f"{out_dir}/candidates_topN.csv"
    write_candidates_csv(topn_path, candidates[:top_n], branch)
    print("Saved:", topn_path)

if debug_run_both_branches and debug_candidates is not None:
    for branch_name, cand_list in debug_candidates.items():
        debug_path = f"{out_dir}/candidates_{branch_name}.csv"
        write_candidates_csv(debug_path, cand_list, branch_name)
        print("Saved:", debug_path)

geojson_path = f"{out_dir}/candidates.geojson"
if transform is not None and crs is not None:
    features = []
    for c in candidates:
        if "x_px_full" in c and "y_px_full" in c:
            x = c["x_px_full"]
            y = c["y_px_full"]
        else:
            x = c.get("x_px", 0.0)
            y = c.get("y_px", 0.0)
        gx, gy = transform * (x, y)
        props = {k: v for k, v in c.items() if k not in ["x_px", "y_px", "x_px_full", "y_px_full"]}
        props["branch"] = branch
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [gx, gy]},
            "properties": props
        })

    geojson = {
        "type": "FeatureCollection",
        "features": features,
        "crs": {"type": "name", "properties": {"name": str(crs)}}
    }

    with open(geojson_path, "w") as f:
        json.dump(geojson, f, indent=2)
    print("Saved:", geojson_path)
else:
    print("No georeference found; GeoJSON not written.")

mask_path = f"{out_dir}/candidates_mask.tif"
try:
    import rasterio
    from rasterio.transform import from_origin
    if transform is None:
        transform = from_origin(0, 0, 1, 1)

    mask_to_write = pit_mask.astype(np.uint8)
    mask_transform = transform

    if branch == "IMAGERY" and detect_downsample_factor > 1:
        if export_fullres_mask:
            mask_to_write = np.repeat(np.repeat(mask_to_write, detect_downsample_factor, axis=0), detect_downsample_factor, axis=1)
            print("Exporting full-res mask (IMAGERY) - may be slow.")
        else:
            mask_transform = transform * transform.scale(detect_downsample_factor, detect_downsample_factor)
            print("Exporting downsampled mask (IMAGERY) for speed.")

    profile = {
        "driver": "GTiff",
        "height": mask_to_write.shape[0],
        "width": mask_to_write.shape[1],
        "count": 1,
        "dtype": "uint8",
        "transform": mask_transform,
        "crs": crs
    }

    with rasterio.open(mask_path, "w", **profile) as dst:
        dst.write(mask_to_write, 1)
    print("Saved:", mask_path)
except Exception as exc:
    print("Rasterio export failed; saving PNG mask. Error:", exc)
    from PIL import Image
    Image.fromarray((pit_mask.astype(np.uint8) * 255)).save(f"{out_dir}/candidates_mask.png")

print("Export time (s):", round(time.time() - t_export, 2))


## 12) Summary

In [ ]:
end_time = time.time()

print("=== Summary ===")
print("Branch:", branch)
print("Image shape:", raw.shape)
print("Candidates:", len(candidates))
print("Runtime (s):", round(end_time - start_time, 2))
print("Output dir:", out_dir)

if len(candidates) == 0:
    print("No candidates detected. Consider relaxing thresholds or checking branch selection.")

## 13) Output inventory


In [ ]:
from pathlib import Path

print("Listing outputs in: ", out_dir)
out_path = Path(out_dir)
if out_path.exists():
    pngs = sorted(out_path.glob('*.png'))
    tifs = sorted(out_path.glob('*.tif'))
    others = sorted(out_path.glob('*.csv')) + sorted(out_path.glob('*.geojson'))
    print("PNGs:", [p.name for p in pngs])
    print("TIFFs:", [p.name for p in tifs])
    print("Other files:", [p.name for p in others])
else:
    print("Output directory not found.")
